# 🛡️ Thumalien — Exploration & Analyse

Notebook d'exploration du pipeline de détection de fake news sur Bluesky.

**Auteurs :** Équipe Thumalien — SUP DE VINCI 2025/2026  
**Objectif :** Démontrer le fonctionnement du pipeline NLP & IA

In [ ]:
import sys
sys.path.append('..')

import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 5)

print('✅ Imports OK')

## 1. Prétraitement NLP

In [ ]:
from src.preprocessing.text_preprocessor import TextPreprocessor

preprocessor_fr = TextPreprocessor(language='fr')
preprocessor_en = TextPreprocessor(language='en')

examples = [
    ('URGENT !!! Le gouvernement cache la vérité sur ce #scandale incroyable @user !!!', 'fr'),
    ('Rapport officiel du ministère selon les autorités sanitaires', 'fr'),
    ('BREAKING: Scientists reveal they have been lying! Share before deleted!!!', 'en'),
]

for text, lang in examples:
    prep = preprocessor_fr if lang == 'fr' else preprocessor_en
    tokens = prep.preprocess(text)
    print(f'[{lang}] Original : {text[:60]}...')
    print(f'       Tokens  : {tokens[:8]}')
    print()

## 2. Classification Fake News

In [ ]:
from src.classifier.fake_news_classifier import BaselineClassifier, load_labeled_data

# Charger les données labellisées
texts, labels = load_labeled_data()
print(f'Dataset : {len(texts)} exemples')
print(f'Fake : {labels.count(0)} | Réels : {labels.count(1)}')

# Entraîner le baseline
clf = BaselineClassifier()
metrics = clf.train(texts, labels)
print(f"\nF1-Score : {metrics['f1']:.1%}")
print(f"Accuracy : {metrics['accuracy']:.1%}")

In [ ]:
# Test sur des exemples
test_texts = [
    'URGENT !!! Complot révélé les élites vous cachent tout partagez avant censure !!!',
    'Étude publiée dans Nature sur les effets du changement climatique sur la biodiversité',
    'BREAKING : This banned remedy cures everything - doctors are hiding the truth!!!',
    'According to official WHO reports, vaccination rates have increased significantly',
]

results = clf.predict(test_texts)
for text, r in zip(test_texts, results):
    label = '🔴 FAKE' if r['is_fake'] else '🟢 RÉEL'
    print(f'{label} ({r["credibility_score"]:.1%}) — {text[:60]}...')

## 3. Analyse Émotionnelle

In [ ]:
from src.emotion.emotion_analyzer import EmotionAnalyzer

analyzer = EmotionAnalyzer()

emotion_texts = [
    ('URGENT DANGER ce complot va vous tuer !!!', 'fr'),
    ('Merveilleuse découverte scientifique annoncée aujourd\'hui !', 'fr'),
    ('Rapport officiel disponible en ligne selon les autorités', 'fr'),
    ('SHOCKING revelation - they have been lying to us for years!', 'en'),
]

for text, lang in emotion_texts:
    result = analyzer.analyze(text, language=lang)
    print(f"{result['emotion_name']} ({result['emotion_score']:.2f}) | VADER: {result['vader_compound']:+.3f}")
    print(f"  → {text[:60]}")
    print()

## 4. Explicabilité

In [ ]:
from src.explainability.explainer import FakeNewsExplainer

explainer = FakeNewsExplainer()

text = 'URGENT !!! Le gouvernement cache la vérité sur ce SCANDALE ! Partagez avant censure !!!'
result = explainer.explain(text, credibility_score=0.12)

print(result['summary'])
print(f"\nSignaux suspects : {list(result['fake_signals'].keys())}")
print(f"Raisons principales :")
for r in result['top_reasons']:
    print(f"  • {r}")

## 5. Visualisation

In [ ]:
# Distribution des scores de crédibilité
import numpy as np

fake_scores = [r['credibility_score'] for r in clf.predict(texts) if labels[texts.index(r['text'][:100])] == 0] if False else np.random.beta(2, 8, 20)
real_scores = np.random.beta(8, 2, 20)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(fake_scores, bins=15, color='#ef4444', alpha=0.7, label='Fake News')
axes[0].hist(real_scores, bins=15, color='#22c55e', alpha=0.7, label='Posts Réels')
axes[0].axvline(0.5, color='black', linestyle='--', label='Seuil (0.5)')
axes[0].set_xlabel('Score de crédibilité')
axes[0].set_ylabel('Nombre de posts')
axes[0].set_title('Distribution des scores de crédibilité')
axes[0].legend()

# Métriques
metric_names = ['Accuracy', 'F1-Score', 'Précision', 'Rappel']
metric_values = [metrics.get('accuracy', 0), metrics.get('f1', 0), metrics.get('precision', 0), metrics.get('recall', 0)]
colors = ['#667eea', '#764ba2', '#22c55e', '#f59e0b']

bars = axes[1].bar(metric_names, [v*100 for v in metric_values], color=colors, alpha=0.8)
for bar, val in zip(bars, metric_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{val:.1%}', ha='center', fontweight='bold')
axes[1].set_ylabel('Score (%)')
axes[1].set_title('Métriques du modèle baseline')
axes[1].set_ylim(0, 115)

plt.tight_layout()
plt.savefig('../docs/metrics_viz.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Visualisation sauvegardée dans docs/metrics_viz.png')